# Этап 1. Ответы модели и её скрытые состояния (локальный запуск)

Этот ноутбук запускается на локальном компьютере с GPU. Убедитесь, что:
- создано виртуальное окружение и установлены зависимости (`pip install -r requirements.txt`);
- CUDA / ROCm видны PyTorch (`torch.cuda.is_available()`);
- RuBQ 2.0 скачан в `data/raw/` и подготовлен `data/questions.jsonl` (`python -m hprobe.data`).

Все результаты пишутся в `outputs/` внутри репозитория. Если прогон прервётся, просто перезапустите ячейку — `generate.py` продолжит с места остановки.

In [ ]:
import os
import sys
import subprocess
import torch
from pathlib import Path

# Путь к корню репозитория (папка, в которой лежит notebooks/)
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))

print('Репозиторий:', REPO.resolve())
print('GPU доступна:', torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Подготовка данных (один раз, CPU)

In [ ]:
# Если data/questions.jsonl ещё не создан, раскомментируйте и запустите:
# !PYTHONPATH=src python -m hprobe.data

## 1. Быстрая проверка на 32 вопросах (1–2 минуты)

In [ ]:
SMOKE = REPO / 'outputs' / 'smoke'
subprocess.run([
    sys.executable, '-m', 'hprobe.generate',
    '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
    '--lang', 'ru',
    '--limit', '32',
    '--out', str(SMOKE),
    '--batch-size', '8',
], cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src'}, check=True)

subprocess.run([
    sys.executable, '-m', 'hprobe.label',
    '--run', str(SMOKE),
], cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src'}, check=True)

import pandas as pd
pd.read_csv(SMOKE / 'labels.csv')[['question', 'answer', 'gold', 'correct', 'consistency']].head(15)

## 2. Основные прогоны

Порядок важен: сначала главная модель (1.5B) на обоих языках, затем 0.5B и 3B для эксперимента с размером модели. Ориентировочно 15–30 минут на прогон 1.5B на GPU с 16 ГБ. Если не хватает памяти, уменьшите `--batch-size`.

In [ ]:
OUT = REPO / 'outputs'
RUNS = [
    ('Qwen/Qwen2.5-1.5B-Instruct', 'qwen2.5-1.5b', 16),
    ('Qwen/Qwen2.5-0.5B-Instruct', 'qwen2.5-0.5b', 32),
    ('Qwen/Qwen2.5-3B-Instruct',   'qwen2.5-3b',   8),
]

for model, short, bs in RUNS:
    for lang in ['ru', 'en']:
        run_out = OUT / short / lang
        print(f'\n=== {model} / {lang} ===')
        subprocess.run([
            sys.executable, '-m', 'hprobe.generate',
            '--model', model,
            '--lang', lang,
            '--out', str(run_out),
            '--batch-size', str(bs),
        ], cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src'}, check=True)
        subprocess.run([
            sys.executable, '-m', 'hprobe.label',
            '--run', str(run_out),
        ], cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src'}, check=True)

## 3. Ручная проверка разметки

В папке каждого прогона появился `manual_check.csv` (100 случайных ответов). Для главного прогона `qwen2.5-1.5b/ru` откройте файл в Excel/LibreOffice, в столбце `human` поставьте 1, если ответ модели верный, и 0, если неверный. Сохраните как CSV на то же место и запустите ячейку ниже: получится точность автоматической разметки — это число пойдёт в тезисы.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'hprobe.label',
    '--run', str(OUT / 'qwen2.5-1.5b' / 'ru'),
    '--check',
], cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src'}, check=True)

## 4. Первые результаты: пробы по слоям

Для каждого прогона обучаются пробы по всем слоям и сравниваются с базовыми методами. GPU не нужен. График `probe/layers.png` и таблица `probe/summary.csv` сохраняются в папке прогона.

In [ ]:
from IPython.display import Image, display

for run in sorted(OUT.glob('*/*')):
    if not (run / 'hidden.npy').exists():
        continue
    print(f'\n--- {run} ---')
    subprocess.run([
        sys.executable, '-m', 'hprobe.probes',
        '--run', str(run),
    ], cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src'}, check=True)
    display(Image(filename=run / 'probe' / 'layers.png', width=600))

## 5. Адаптивный поиск (опционально, GPU)

Для лучшей модели/языка строится кривая «точность vs доля вопросов с поиском». Абзацы берутся из RuBQ.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'hprobe.search',
    '--run', str(OUT / 'qwen2.5-1.5b' / 'ru'),
    '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--lang', 'ru',
], cwd=REPO, env={**os.environ, 'PYTHONPATH': 'src'}, check=True)

display(Image(filename=OUT / 'qwen2.5-1.5b' / 'ru' / 'search' / 'curve.png', width=500))